# RAG with Docling, LangChain, and ChromaDB

[**Open in Colab**](https://colab.research.google.com/github/HassanAlgoz/B5/blob/main/content/W4/M3/lessons/05_rag_docling.ipynb)

| Step | Tech | Execution |
| --- | --- | --- |
| Parsing | Docling |  💻 Local |
| Embedding | HuggingFace |  💻 Local |
| Vector store | ChromaDB | 💻 Local |
| LLM | OpenRouter | 🌐 Remote |

## Colab Setup

It is recommended to use GPU-enabled runtime.

### Install dependencies

In [ ]:
%pip install -q --progress-bar off --no-warn-conflicts \
    langchain-docling langchain-core langchain-huggingface \
    langchain_community langchain-openai langchain-chroma chromadb \
    langgraph langchain python-dotenv rich requests

In [ ]:
import json
import os
from textwrap import dedent

from rich import print

### Environment variables

In [3]:
from dotenv import load_dotenv

def _get_env_from_colab_or_os(key: str) -> str | None:
    """Read an environment variable from Colab secrets first, then OS env."""
    try:
        from google.colab import userdata
        try:
            return userdata.get(key)
        except userdata.SecretNotFoundError:
            pass
    except ImportError:
        pass
    return os.getenv(key)

In [ ]:
load_dotenv()

In [ ]:
# Disable tokenizer parallel workers to avoid noisy notebook warnings.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 1. Document loading and chunking

Now we can instantiate our loader and load documents.

This section loads and chunks the source document using Docling so we can build retrieval units for the RAG pipeline.

In [ ]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer
from langchain_docling.loader import ExportType


# Source document for the lesson; small and publicly available for reproducible demos.
file_path = ["https://arxiv.org/pdf/2408.09869"]

# Same as embedding model.
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"

# Export mode used by this lesson; DOC_CHUNKS gives retrieval-ready chunks directly.
export_type = ExportType.DOC_CHUNKS

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(embed_model_id)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [ ]:
from docling.chunking import HybridChunker
from langchain_docling import DoclingLoader

# from docling.datamodel.pipeline_options import PdfPipelineOptions

# pipeline_options = PdfPipelineOptions()
# enable third-party plugins (like langchain_docling)
# pipeline_options.allow_external_plugins = True  

loader = DoclingLoader(
    file_path=file_path,
    export_type=export_type,
    chunker=HybridChunker(tokenizer=tokenizer),
    # convert_kwargs={"pipeline_options": pipeline_options},
)

docs = loader.load()

[INFO] 2026-04-28 16:37:49,745 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-28 16:37:49,752 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-28 16:37:51,063 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-04-28 16:37:51,179 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-28 16:37:51,182 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-28 16:37:51,395 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-28 16:37:51,397 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-04-28 16:37:51,785 [

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (587 > 512). Running this sequence through the model will result in indexing errors


Let's see how many chunks we have:

In [9]:
(type(docs), len(docs))

(list, 50)

Let's take a look at the content of the first few chunks:

In [10]:
print(docs[1].page_content)

Abstract
This technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF 
document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and 
table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. 
The code interface allows for easy extensibility and addition of new features and models.

In [11]:
print(docs[2].page_content)

1 Introduction
Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their
huge variability in formats, weak standardization and printing-optimized characteristic, which discards most 
structural features and metadata. With the advent of LLMs and popular application patterns such as 
retrieval-augmented generation (RAG), leveraging the rich content embedded in PDFs has become ever more relevant. 
In the past decade, several powerful document understanding solutions have emerged on the market, most of which are
commercial software, cloud offerings [3] and most recently, multi-modal vision-language models. As of today, only a
handful of open-source tools cover PDF conversion, leaving a significant feature and quality gap to proprietary 
solutions.
With Docling , we open-source a very capable and efficient document conversion tool which builds on the powerful, 
specialized AI models and datasets for layout analysis and table structure recognition we developed and presented 
in the recent past [12, 13, 9]. Docling is designed as a simple, self-contained python library with permissive 
license, running entirely locally on commodity hardware. Its code architecture allows for easy extensibility and 
addition of new features and models.
Here is what Docling delivers today:

In [12]:
print(docs[3].page_content)

1 Introduction
- Converts PDF documents to JSON or Markdown format, stable and lightning fast
- Understands detailed page layout, reading order, locates figures and recovers table structures
- Extracts metadata from the document, such as title, authors, references and language
- Optionally applies OCR, e.g. for scanned PDFs
- Can be configured to be optimal for batch-mode (i.e high throughput, low time-to-solution)
- or interactive mode (compromise on efficiency, low time-to-solution)
- Can leverage different accelerators (GPU, MPS, etc).

In [13]:
print(docs[4].page_content)

2 Getting Started
To use Docling, you can simply install the docling package from PyPI. Documentation and examples are available in 
our GitHub repository at github.com/DS4SD/docling. All required model assets 1 are downloaded to a local 
huggingface datasets cache on first use, unless you choose to pre-install the model assets in advance.
Docling provides an easy code interface to convert PDF documents from file system, URLs or binary streams, and 
retrieve the output in either JSON or Markdown format. For convenience, separate methods are offered to convert 
single documents or batches of documents. A basic usage example is illustrated below. Further examples are 
available in the Doclign code repository.
from docling.document_converter import DocumentConverter

In [14]:
print(docs[5].page_content)

2 Getting Started
```
source = "https://arxiv.org/pdf/2206.01062" # PDF path or URL converter = DocumentConverter() result = 
converter.convert_single(source) print(result.render_as_markdown()) # output: "## DocLayNet: A Large Human 
-Annotated Dataset for Document -Layout Analysis [...]"
```
Optionally, you can configure custom pipeline features and runtime options, such as turning on or off features 
(e.g. OCR, table structure recognition), enforcing limits on the input document size, and defining the budget of 
CPU threads. Advanced usage examples and options are documented in the README file. Docling also provides a 
Dockerfile to demonstrate how to install and run it inside a container.

## 2. Indexing

### 2.1 Embedding Model

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name=embed_model_id)

### 2.2 Vector Store

In [21]:
from pathlib import Path
import shutil
import tempfile
import uuid

persist_directory = Path(tempfile.gettempdir()) / f"chroma_db_docling_{uuid.uuid4().hex[:8]}"
if persist_directory.exists():
    # Best-effort cleanup on reruns.
    shutil.rmtree(persist_directory, ignore_errors=True)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

vectorstore = Chroma.from_documents(
    documents=filter_complex_metadata(docs),
    embedding=embedding,
    collection_name="docling_demo",
    persist_directory=str(persist_directory),
)

## 3. Retrieval and Generation

### 3.1 Retriever

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### 3.2 LLM

In [ ]:
from langchain_openai import ChatOpenAI

# OpenRouter chat model
openrouter_llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=_get_env_from_colab_or_os("OPENROUTER_API_KEY"),
)

Alternatively, one could use HuggingFace servers and run an open-source model there:

In [25]:
# Hugging Face Inference API integrated directly as a LangChain model.
# from langchain_huggingface import HuggingFaceEndpoint

# hf_llm = HuggingFaceEndpoint(
#     repo_id="mistralai/Mixtral-8x7B-Instruct-v0.1",
#     huggingfacehub_api_token=_get_env_from_colab_or_os("HF_TOKEN"),
#     task="text-generation",
# )

### 3.3 Tasks

In [26]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.func import entrypoint, task

In [27]:
@task
def retrieve(query: str):
    """Fetch top-k relevant documents from the vector store retriever."""
    return retriever.invoke(query)

In [28]:
@task
def generate(query: str, passages: list[str], use_openrouter: bool = True) -> str:
    """Generate an answer from retrieved passages using a selected LangChain LLM."""
    system_text = dedent("""
        Role: You are a helpful and informative assistant. Answer using the reference passages below.
        Style and Tone: Respond in complete sentences for a non-technical audience in a friendly, conversational tone.
        Note: If a passage is irrelevant, you may ignore it.
    """)
    passages_serialized = "\n".join(f"<passage>{p}</passage>" for p in passages)
    user_text = dedent(f"""
        <question>{query}</question>
        <passages>
        {passages_serialized}
        </passages>
    """)

    if use_openrouter:
        msg = openrouter_llm.invoke([
            SystemMessage(system_text),
            HumanMessage(user_text)
        ])
        return msg.content

    # prompt = f"{system_text}\n\n{user_text}"
    # return hf_llm.invoke(prompt)

### 3.4 Workflow Entrypoint

In [29]:
@entrypoint()
def rag_qa_workflow(query: str, use_openrouter: bool = True) -> dict:
    """Run retrieval and generation as a functional LangGraph RAG workflow."""
    docs = retrieve(query).result()
    answer = generate(query, [doc.page_content for doc in docs], use_openrouter).result()
    return {"question": query, "answer": answer, "context": docs}

## 4. Running the Workflow

This cell executes the functional RAG workflow end-to-end (retrieve then generate), prints a clipped final answer, and shows the source chunks used for grounding.

In [30]:
# Demo query chosen to match the Docling report section headers for clear retrieval traces.
question = "Which are the main AI models in Docling?"
d = rag_qa_workflow.invoke(question, use_openrouter=True)

In [ ]:
def clip_text(text: str, threshold: int = 100) -> str:
    """Return a truncated preview of long text for cleaner notebook output."""
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [31]:
clipped_answer = clip_text(d["answer"], threshold=200)
print(f"Question:\n{d['question']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(d["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")

Question:
Which are the main AI models in Docling?

Answer:
The main AI models that Docling ships are:

1. **A layout‑analysis model** – it works like an object detector, identifying and locating different page elements
(such as headings, paragraphs, images, e...

Source 1:

text: "3.2 AI models\nAs part of Docling, we initially release two highly capable AI models to the open-source 
community, which have been developed and published recently by our team. The first model is a layout analysis 
model, an accurate object-detector for page elements [13]. The second model is TableFormer [12, 9], a 
state-of-the-art table structure re..."

source: https://arxiv.org/pdf/2408.09869

Source 2:

text: "3 Processing pipeline\nDocling implements a linear pipeline of operations, which execute sequentially on 
each given document (see Fig. 1). Each document is first parsed by a PDF backend, which retrieves the programmatic 
text tokens, consisting of string content and its coordinates on the page, and also renders a bitmap image of each 
page to support ..."

source: https://arxiv.org/pdf/2408.09869

Source 3:

text: "6 Future work and contributions\nDocling is designed to allow easy extension of the model library and 
pipelines. In the future, we plan to extend Docling with several more models, such as a figure-classifier model, an
equationrecognition model, a code-recognition model and more. This will help improve the quality of conversion for 
specific types of ..."

source: https://arxiv.org/pdf/2408.09869